# Interactive data story (Plotly)

Interactive views of the same themes covered in **`country_analysis.ipynb`** and **`category_analysis.ipynb`**, with **hover tooltips**, **zoom/pan**, and **export to HTML** for your portfolio or report.

**Install once:** `pip install plotly`

Optional static PNG export: `pip install kaleido` then `fig.write_image("plot.png")`.

**World map:** see `datastory_map.ipynb` — the map is exported by `map_ui_export.py` to **`exploration_images/datastory_interactive_map_controls.html`** (buttons for category / engagement / publication time UTC, with dropdowns or an hour slider). A copy is also written to `exploration_images/datastory_interactive_map.html`.


In [ ]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = "plotly_white"

OUT_DIR = "exploration_images"

COUNTRY_ORDER = ["US", "GB", "CA", "DE", "FR", "IN", "KR", "MX", "JP", "RU"]
COUNTRY_FULL_NAME = {
    "US": "United States",
    "GB": "United Kingdom",
    "CA": "Canada",
    "DE": "Germany",
    "FR": "France",
    "IN": "India",
    "KR": "South Korea",
    "MX": "Mexico",
    "JP": "Japan",
    "RU": "Russia",
}


def country_full(code: str) -> str:
    return COUNTRY_FULL_NAME.get(str(code), str(code))


SPARSE_CATEGORY_NAMES = {"Movies", "Trailers", "Nonprofits & Activism", "Shows", "Short Movies"}
MIN_GLOBAL_CATEGORY = 4000
MIN_CELL_COUNT = 120

data = "../data/"
df_us = pd.read_csv(data + "USvideos.csv")
df_ca = pd.read_csv(data + "CAvideos.csv")
df_de = pd.read_csv(data + "DEvideos.csv")
df_fr = pd.read_csv(data + "FRvideos.csv")
df_gb = pd.read_csv(data + "GBvideos.csv")
df_in = pd.read_csv(data + "INvideos.csv")
df_jp = pd.read_csv(data + "JPvideos.csv", encoding="utf-8-sig", encoding_errors="replace", engine="python")
df_kr = pd.read_csv(data + "KRvideos.csv", encoding="utf-8-sig", encoding_errors="replace", engine="python")
df_mx = pd.read_csv(data + "MXvideos.csv", encoding="utf-8-sig", encoding_errors="replace", engine="python")
df_ru = pd.read_csv(data + "RUvideos.csv", encoding="utf-8-sig", encoding_errors="replace", engine="python")

for d, c in [
    (df_us, "US"), (df_ca, "CA"), (df_de, "DE"), (df_fr, "FR"), (df_gb, "GB"),
    (df_in, "IN"), (df_jp, "JP"), (df_kr, "KR"), (df_mx, "MX"), (df_ru, "RU"),
]:
    d["country"] = c

df = pd.concat(
    [df_us, df_ca, df_de, df_fr, df_gb, df_in, df_jp, df_kr, df_mx, df_ru],
    ignore_index=True,
).drop_duplicates()

category_map = {
    1: "Film & Animation", 2: "Autos & Vehicles", 10: "Music",
    15: "Pets & Animals", 17: "Sports", 18: "Short Movies",
    19: "Travel & Events", 20: "Gaming", 21: "Videoblogging",
    22: "People & Blogs", 23: "Comedy", 24: "Entertainment",
    25: "News & Politics", 26: "Howto & Style", 27: "Education",
    28: "Science & Technology", 29: "Nonprofits & Activism",
    30: "Movies", 43: "Shows", 44: "Trailers",
}
df["category"] = df["category_id"].map(category_map).fillna("Other")

df["publish_time"] = pd.to_datetime(df["publish_time"], errors="coerce").dt.tz_localize(None)
df["publish_hour"] = df["publish_time"].dt.hour

df["comment_rate"] = df["comment_count"] / df["views"]
df["like_dislike_ratio"] = df["likes"] / (df["dislikes"] + 1)

cat_totals = df.groupby("category").size()

LAYOUT_BASE = dict(
    font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", size=13),
    margin=dict(l=60, r=40, t=80, b=60),
    hoverlabel=dict(bgcolor="white", font_size=13),
)

print("Ready:", len(df), "rows")


## 1. Category mix by country (interactive heatmap)


In [ ]:
cat_country = df.groupby(["country", "category"]).size().unstack(fill_value=0)
cat_country = cat_country.reindex([c for c in COUNTRY_ORDER if c in cat_country.index])
cat_country_pct = cat_country.div(cat_country.sum(axis=1), axis=0) * 100

top_cats = cat_country.sum().nlargest(10).index.tolist()
heat_pct = cat_country_pct[top_cats]
heat_raw = cat_country[top_cats]

xlabels = []
for c in heat_pct.columns:
    flag = " †" if cat_totals.get(c, 0) < MIN_GLOBAL_CATEGORY or c in SPARSE_CATEGORY_NAMES else ""
    xlabels.append((c[:20] + "…") if len(c) > 22 else c + flag)

hover = []
for i, country in enumerate(heat_pct.index):
    row = []
    cn = country_full(country)
    for j, cat in enumerate(heat_pct.columns):
        pct = heat_pct.iloc[i, j]
        n = int(heat_raw.iloc[i, j])
        warn = (
            "<br><i>Low sample — interpret cautiously</i>"
            if n < MIN_CELL_COUNT
            else ""
        )
        row.append(
            f"{cn}<br>{cat}<br><b>Share: {pct:.1f}%</b>{warn}"
        )
    hover.append(row)

fig1 = go.Figure(
    data=go.Heatmap(
        z=heat_pct.values,
        x=xlabels,
        y=[country_full(c) for c in heat_pct.index],
        colorscale="YlOrRd",
        hoverinfo="text",
        text=hover,
        hovertemplate="%{text}<extra></extra>",
        colorbar=dict(title=dict(text="% of country’s rows", side="right")),
    )
)
fig1.update_layout(
    **LAYOUT_BASE,
    title=dict(
        text="<b>Category mix by country</b><br><sup>Top 10 categories by global volume · † = sparse category globally</sup>",
        x=0.5,
        xanchor="center",
    ),
    xaxis=dict(tickangle=-35),
    yaxis=dict(autorange="reversed", title="Country"),
    height=520,
    width=980,
)
fig1.show(config={"displayModeBar": True, "displaylogo": False, "scrollZoom": True})
fig1.write_html(f"{OUT_DIR}/datastory_interactive_01_category_mix.html", include_plotlyjs="cdn", full_html=True)
print(f"Saved {OUT_DIR}/datastory_interactive_01_category_mix.html")


## 2. Engagement — comment rate & like/dislike ratio


In [ ]:
def median_by_country_order(d, col):
    return d.groupby("country")[col].median().reindex([c for c in COUNTRY_ORDER if c in d["country"].values])


cr = median_by_country_order(df, "comment_rate")
ldr = median_by_country_order(df, "like_dislike_ratio")

fig2 = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Median comment rate (comments / view)",
        "Median like / dislike ratio",
    ),
    horizontal_spacing=0.14,
)

fig2.add_trace(
    go.Bar(
        x=cr.values,
        y=[country_full(c) for c in cr.index],
        orientation="h",
        marker_color="steelblue",
        hovertemplate="%{y}<br><b>Median: %{x:.5f}</b><extra></extra>",
    ),
    row=1,
    col=1,
)
fig2.add_trace(
    go.Bar(
        x=ldr.values,
        y=[country_full(c) for c in ldr.index],
        orientation="h",
        marker_color="#6a4c93",
        hovertemplate="%{y}<br><b>Median: %{x:.2f}</b><extra></extra>",
    ),
    row=1,
    col=2,
)

fig2.update_layout(
    **LAYOUT_BASE,
    title=dict(text="<b>Engagement per view</b>", x=0.5, xanchor="center"),
    showlegend=False,
    height=440,
    width=1000,
)
fig2.update_yaxes(autorange="reversed", row=1, col=1)
fig2.update_yaxes(autorange="reversed", row=1, col=2)
fig2.show(config={"displayModeBar": True, "displaylogo": False})
fig2.write_html(f"{OUT_DIR}/datastory_interactive_02_engagement.html", include_plotlyjs="cdn", full_html=True)
print(f"Saved {OUT_DIR}/datastory_interactive_02_engagement.html")


## 3. Publish hour (UTC) — interactive heatmap

Hours are **UTC** (from `publish_time`), not local wall time.


In [ ]:
hour_country = (
    df.groupby(["country", "publish_hour"]).size().unstack(fill_value=0)
)
hour_country = hour_country.reindex([c for c in COUNTRY_ORDER if c in hour_country.index])
hour_country_pct = hour_country.div(hour_country.sum(axis=1), axis=0) * 100

hours = [str(h) for h in hour_country_pct.columns]

hover_h = []
for i, country in enumerate(hour_country_pct.index):
    row = []
    cn = country_full(country)
    for j, h in enumerate(hour_country_pct.columns):
        zv = hour_country_pct.iloc[i, j]
        row.append(f"{cn}<br>Hour {int(h)} UTC<br><b>Share: {zv:.2f}%</b>")
    hover_h.append(row)

fig3 = go.Figure(
    data=go.Heatmap(
        z=hour_country_pct.values,
        x=hours,
        y=[country_full(c) for c in hour_country_pct.index],
        colorscale="YlGnBu",
        hoverinfo="text",
        text=hover_h,
        hovertemplate="%{text}<extra></extra>",
        colorbar=dict(title=dict(text="% within country", side="right")),
    )
)
fig3.update_layout(
    **LAYOUT_BASE,
    title=dict(
        text=(
            "<b>Publish hour by country</b><br>"
            "<sup>Share within each country · All hours are <b>UTC</b> (not local time)</sup>"
        ),
        x=0.5,
        xanchor="center",
    ),
    xaxis=dict(title="Hour of day (UTC)", tickmode="linear", dtick=2),
    yaxis=dict(autorange="reversed", title="Country"),
    height=520,
    width=1100,
)
fig3.show(config={"displayModeBar": True, "displaylogo": False, "scrollZoom": True})
fig3.write_html(f"{OUT_DIR}/datastory_interactive_03_publish_hour_utc.html", include_plotlyjs="cdn", full_html=True)
print(f"Saved {OUT_DIR}/datastory_interactive_03_publish_hour_utc.html")


## 4. Country share within each category


In [ ]:
country_cat = df.groupby(["category", "country"]).size().unstack(fill_value=0)
country_cat_pct = country_cat.div(country_cat.sum(axis=1), axis=0) * 100

eligible_cats = cat_totals[cat_totals >= MIN_GLOBAL_CATEGORY].index.tolist()
if len(eligible_cats) < 8:
    eligible_cats = cat_totals.head(12).index.tolist()

country_cat_pct = country_cat_pct.loc[eligible_cats]
country_cat_raw = country_cat.loc[eligible_cats]

vol_country = df["country"].value_counts()
top_countries = sorted(
    vol_country.nlargest(8).index.tolist(),
    key=lambda c: COUNTRY_ORDER.index(c) if c in COUNTRY_ORDER else 99,
)

sub_pct = country_cat_pct[top_countries]
sub_raw = country_cat_raw[top_countries]

hover4 = []
for i, cat in enumerate(sub_pct.index):
    row = []
    for j, country in enumerate(sub_pct.columns):
        pct = sub_pct.iloc[i, j]
        n = int(sub_raw.iloc[i, j])
        cfull = country_full(country)
        warn = (
            "<br><i>Low sample — interpret cautiously</i>"
            if n < MIN_CELL_COUNT
            else ""
        )
        row.append(f"{cat}<br>{cfull}<br><b>Share: {pct:.1f}%</b>{warn}")
    hover4.append(row)

ycat = [c[:40] + "…" if len(c) > 42 else c for c in sub_pct.index]

fig4 = go.Figure(
    data=go.Heatmap(
        z=sub_pct.values,
        x=[country_full(c) for c in sub_pct.columns],
        y=ycat,
        colorscale="YlGnBu",
        hoverinfo="text",
        text=hover4,
        hovertemplate="%{text}<extra></extra>",
        colorbar=dict(title=dict(text="% of category’s rows", side="right")),
    )
)
fig4.update_layout(
    **LAYOUT_BASE,
    title=dict(
        text=(
            "<b>Country mix within each category</b><br>"
            f"<sup>Categories with ≥{MIN_GLOBAL_CATEGORY} global rows (or fallback)</sup>"
        ),
        x=0.5,
        xanchor="center",
    ),
    xaxis=dict(title="Country"),
    yaxis=dict(autorange="reversed", title="Category", tickfont=dict(size=11)),
    height=max(480, 44 * len(sub_pct)),
    width=920,
)
fig4.show(config={"displayModeBar": True, "displaylogo": False, "scrollZoom": True})
fig4.write_html(f"{OUT_DIR}/datastory_interactive_04_country_within_category.html", include_plotlyjs="cdn", full_html=True)
print(f"Saved {OUT_DIR}/datastory_interactive_04_country_within_category.html")


---

### One-page HTML (all figures)

The next cell concatenates the four charts into **`exploration_images/datastory_interactive_dashboard.html`** (one scrollable page; Plotly loads once from CDN).


In [ ]:
cfg = dict(displayModeBar=True, displaylogo=False, scrollZoom=True)

blocks = [
    "<h1>Trending YouTube — interactive data story</h1>",
    "<p><strong>Note:</strong> publish-hour chart uses <strong>UTC</strong> hours.</p>",
    "<h2>1. Category mix by country</h2>",
    pio.to_html(fig1, include_plotlyjs="cdn", full_html=False, config=cfg),
    "<h2>2. Engagement (comment rate &amp; like/dislike ratio)</h2>",
    pio.to_html(fig2, include_plotlyjs=False, full_html=False, config=cfg),
    "<h2>3. Publish hour (UTC)</h2>",
    pio.to_html(fig3, include_plotlyjs=False, full_html=False, config=cfg),
    "<h2>4. Country share within each category</h2>",
    pio.to_html(fig4, include_plotlyjs=False, full_html=False, config=cfg),
]

page = f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1" />
<title>Trending YouTube — data story</title>
<style>
body {{ font-family: system-ui, -apple-system, Segoe UI, sans-serif; max-width: 1080px; margin: 0 auto; padding: 1rem 1.25rem 3rem; line-height: 1.45; }}
h1 {{ font-size: 1.65rem; }}
h2 {{ font-size: 1.15rem; margin-top: 2.25rem; margin-bottom: 0.5rem; border-bottom: 1px solid #e0e0e0; padding-bottom: 0.35rem; }}
</style></head><body>
{''.join(blocks)}
</body></html>"""

out_path = f"{OUT_DIR}/datastory_interactive_dashboard.html"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(page)
print(f"Saved {out_path}")
